In [0]:
%run ../init_framework

In [0]:
# 1. STREAM INIT
# startingVersion 1 is mandatory for the initial bootstrap of the silver layer.
df_bronze_defaulters = (spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 1) 
    .table(DEFAULTERS_BRONZE))    

In [0]:
# 2. METADATA ENRICHMENT
# Injects standard audit columns (_ingested_at, etc.)
df_with_metadata = apply_silver_metadata(df_bronze_defaulters)

In [0]:
# 3. INITIAL DEDUPLICATION
# Global distinct to handle exact row duplicates if upstream guarantees fail.
df_distinct = df_with_metadata.distinct()

In [0]:
from pyspark.sql.functions import col

# 4. DELINQUENCY FIX
# Fixing the delinquency column to be an integer 
    # -- Cast to float first to avoid losing decimal strings to NULL
    # -- Cast to int to perform the "floor" truncation
    # -- Replace all NULLs (original nulls + invalid strings) with 0

df_delinq2yrs_fixed = df_distinct.withColumn(
    "delinq_2yrs", col("delinq_2yrs").cast("float").cast("int")
).fillna(0, subset=["delinq_2yrs"]) 

In [0]:
# 5. DELINQUENCY ENTITY
from pyspark.sql.functions import col

# filter logic: only keep records where there's actually a delinquency event to track.
df_defaulters_delinq = (df_delinq2yrs_fixed
    .select(
        "member_id",
        "delinq_2yrs",
        "delinq_amnt",
        col("mths_since_last_delinq").cast("int"),
        # Metadata Columns
        "_ingested_at",
        "_source_file",
        "_bronze_version",
        "_updated_at"
    )
    .filter((col("delinq_2yrs") > 0) | (col("mths_since_last_delinq") > 0))
)

In [0]:
# 6. SINK: DELINQUENCY
# final dedup on PK to avoid merge collisions.
df_defaulters_delinq_final = df_defaulters_delinq.dropDuplicates(["member_id"])

def upsert_defaulters_delinq_to_silver(microBatchDF, batchId):
    # standard merge with version fencing to protect against out-of-order data.
    spark_session = microBatchDF.sparkSession
    microBatchDF.createOrReplaceTempView("defaulters_delinq_batch_updates")

    merge_query = f"""
    MERGE INTO {DEFAULTERS_DELINQ_SILVER} AS target
        USING defaulters_delinq_batch_updates AS source
        ON target.member_id = source.member_id
        WHEN MATCHED AND source._bronze_version > target._bronze_version THEN
          UPDATE SET
            target.delinq_2yrs = source.delinq_2yrs,
            target.delinq_amnt = source.delinq_amnt,
            target.mths_since_last_delinq = source.mths_since_last_delinq,
            target._ingested_at = source._ingested_at,
            target._source_file = source._source_file,
            target._bronze_version = source._bronze_version,
            target._updated_at = source._updated_at
        WHEN NOT MATCHED THEN
          INSERT (
            member_id, delinq_2yrs, delinq_amnt, mths_since_last_delinq,
            _ingested_at, _source_file, _bronze_version, _updated_at
          )
          VALUES (
            source.member_id, source.delinq_2yrs, source.delinq_amnt, source.mths_since_last_delinq,
            source._ingested_at, source._source_file, source._bronze_version, source._updated_at
          )
    """
    spark_session.sql(merge_query)

# tuning for local 8-core cluster
spark.conf.set("spark.sql.shuffle.partitions", "16")

# Execute trigger-once pipeline for batch-like efficiency on streaming logic.
query_defaulters_delinq = (
    df_defaulters_delinq_final.writeStream
    .outputMode("append") 
    .option("checkpointLocation", SILVER_CHECKPOINT_DEFAULTERS_DELINQ)
    .trigger(availableNow=True)
    .foreachBatch(upsert_defaulters_delinq_to_silver)
    .start()
)

# block notebook termination until micro-batch is committed
query_defaulters_delinq.awaitTermination()

In [0]:
%sql
select count(1) from lending_risk.silver.defaulters_delinq;
-- select * from lending_risk.silver.defaulters_delinq limit 3;
-- 733
-- 1484

In [0]:
# 7. ENQUIRY ENTITY
from pyspark.sql.functions import col

# pull inquiry fields for records with any history of public records or inquiries.
df_defaulters_inquiry = (df_delinq2yrs_fixed
    .select(
        "member_id", 
        "pub_rec", 
        "pub_rec_bankruptcies", 
        "inq_last_6mths",
        # Metadata Columns
        "_ingested_at",
        "_source_file",
        "_bronze_version",
        "_updated_at"
    )
    .filter(
        (col("pub_rec") > 0.0) | 
        (col("pub_rec_bankruptcies") > 0.0) | 
        (col("inq_last_6mths") > 0.0)
    )
)

In [0]:
# 8. SINK: ENQUIRIES
df_defaulters_inquiry_final = df_defaulters_inquiry.dropDuplicates(["member_id"])

def upsert_defaulters_inquiry_to_silver(microBatchDF, batchId):
    spark_session = microBatchDF.sparkSession
    microBatchDF.createOrReplaceTempView("defaulters_inquiry_batch_updates")

    merge_query = f"""
    MERGE INTO {DEFAULTERS_INQUIRY_SILVER} AS target
        USING defaulters_inquiry_batch_updates AS source
        ON target.member_id = source.member_id
        WHEN MATCHED AND source._bronze_version > target._bronze_version THEN
          UPDATE SET
            target.pub_rec = source.pub_rec,
            target.pub_rec_bankruptcies = source.pub_rec_bankruptcies,
            target.inq_last_6mths = source.inq_last_6mths,
            target._ingested_at = source._ingested_at,
            target._source_file = source._source_file,
            target._bronze_version = source._bronze_version,
            target._updated_at = source._updated_at
        WHEN NOT MATCHED THEN
          INSERT (
            member_id, pub_rec, pub_rec_bankruptcies, inq_last_6mths,
            _ingested_at, _source_file, _bronze_version, _updated_at
          )
          VALUES (
            source.member_id, source.pub_rec, source.pub_rec_bankruptcies, source.inq_last_6mths,
            source._ingested_at, source._source_file, source._bronze_version, source._updated_at
          )
    """
    spark_session.sql(merge_query)

query_defaulters_inquiry = (
    df_defaulters_inquiry_final.writeStream
    .outputMode("append") 
    .option("checkpointLocation", SILVER_CHECKPOINT_DEFAULTERS_INQUIRY)
    .trigger(availableNow=True)
    .foreachBatch(upsert_defaulters_inquiry_to_silver)
    .start()
)

query_defaulters_inquiry.awaitTermination()

In [0]:
%sql
select count(1) from lending_risk.silver.defaulters_inquiry;
-- select * from lending_risk.silver.defaulters_inquiry limit 3;
-- 716
-- 1452